[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C33_Context_Memory_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身（MockLLM）

本课全程 **纯 Python 标准库、CPU 可跑、无需任何 API key**（连 pip 都不用）。凡是需要「模型」的地方都用 **MockLLM**——一个确定性的假模型——再用 `assert` 验证 scaffold 逻辑。

这个 notebook 做四件事：① 确认环境；② 建立 **上下文窗口 = 有限预算** 的心智模型；③ 造出本课的主角 **MockLLM**；④ 立下全课纪律——**对拍 / 不变量 + assert**，并演示「**有 key 用真实 Claude、没 key 自动回退 MockLLM**」的适配骨架。

## 1 · 环境自检

只需要 Python 标准库。**全程不联网、不需要 API key、不需要 pip 安装。**

In [ ]:
import sys, platform, json, re, hashlib, math
print('Python', sys.version.split()[0], '|', platform.system())
print('用到的标准库: json, re, hashlib, math —— 全部内置，无需安装')
try:
    import anthropic  # 可选：仅当你想接真实 Claude
    print('anthropic SDK 已安装（可选，用于真实 API；没有也不影响本课）')
except Exception:
    print('anthropic 未安装 —— 完全没关系，本课用 MockLLM 端到端跑通')
print('无需 API key —— 本课用 MockLLM。环境就绪 ✅')

## 2 · 上下文窗口 = 有限预算（最小骨架）

模型一次只能看到 **窗口** 那么多 token。所有要让模型看到的东西——系统提示、工具、记忆、检索、历史、本轮输入——都要挤进这一块，**还要给输出（`max_tokens`）留空间**：`输入 token + max_tokens ≤ 窗口`。

先写一个最小的「预算检查器」：给定各部分的 token 估计与窗口，判断塞不塞得下、还剩多少。

In [ ]:
def fits_budget(parts, window, reserve_output):
    '''parts: {名称: token数}; window: 窗口上限; reserve_output: 给输出预留的 token。
       返回 (是否塞得下, 输入总token, 剩余可用token)。'''
    used = sum(parts.values())
    budget_for_input = window - reserve_output   # 必须先扣输出预留！
    fits = used <= budget_for_input
    remaining = budget_for_input - used
    return fits, used, remaining

parts = {'system': 500, 'tools': 800, 'history': 3000, 'user_input': 200}
fits, used, remaining = fits_budget(parts, window=8000, reserve_output=1000)
print(f'输入共 {used} tokens, 窗口 8000, 预留输出 1000 -> 可用 {8000-1000}')
print(f'塞得下吗? {fits}; 还剩 {remaining} tokens 可放检索/记忆')
assert fits is True and used == 4500 and remaining == 2500
# 反例：历史涨到 7000 就会爆
fits2, _, rem2 = fits_budget({**parts, 'history': 7000}, window=8000, reserve_output=1000)
assert fits2 is False and rem2 < 0, '超预算应被检测到'
print('✅ 预算检查器：先扣输出预留，再判断输入塞不塞得下 —— 这就是全课的地基')

## 3 · 造出本课的主角：MockLLM

真实上下文管线里，决定「摘要长什么样、回答是什么、文本的向量是什么」的是大模型。本课用 **MockLLM** 代替它：一个**确定性、规则驱动**的假模型，对给定输入返回**结构正确、可预测**的输出。这样模型质量的不确定性被剥离，剩下的全是我们 scaffold 的对错——最适合学预算与控制流。

下面这个 MockLLM 提供三种确定性能力：`complete`（按关键词查表回话）、`summarize`（抽取并拼接要点当摘要）、`embed`（确定性词频向量）。形状刻意贴近真实用法。

In [ ]:
class MockLLM:
    '''确定性假模型：不学习、不联网，只按规则把输入映射到可预测输出。
       三个能力刻意对应真实用途：对话补全 / 摘要 / 文本嵌入。'''
    def __init__(self, rules=None, default='(no rule matched)'):
        self.rules = rules or []
        self.default = default
        self.calls = 0
    def complete(self, messages):
        '''看最后一条消息文本, 命中关键词就回固定话。'''
        self.calls += 1
        last = messages[-1]['content'] if messages else ''
        text = last if isinstance(last, str) else json.dumps(last, ensure_ascii=False)
        for kw, resp in self.rules:
            if kw in text:
                return resp
        return self.default
    def summarize(self, text, max_points=3):
        '''确定性摘要: 取前 max_points 个非空行的首句, 拼成要点。'''
        lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
        pts = [ln.split('。')[0].split('. ')[0][:40] for ln in lines[:max_points]]
        return '摘要: ' + '; '.join(pts)
    def embed(self, text, dim=16):
        '''确定性嵌入: 把 token 哈希到 dim 维词频向量(可重复、可断言)。'''
        vec = [0.0] * dim
        for tok in re.findall(r'[\w]+', text.lower()):
            vec[int(hashlib.md5(tok.encode()).hexdigest(), 16) % dim] += 1.0
        return vec

llm = MockLLM(rules=[('天气', '今天晴，26°C。'), ('你好', '你好，我能帮你。')])
print('补全:', llm.complete([{'role':'user','content':'北京天气如何？'}]))
print('摘要:', llm.summarize('用户要查天气。\n助手回答了北京。\n用户表示满意。'))
v1, v2 = llm.embed('天气 北京'), llm.embed('天气 北京')
print('嵌入维度:', len(v1), '| 同输入同向量?', v1 == v2)
assert llm.complete([{'role':'user','content':'问天气'}]) == '今天晴，26°C。'
assert llm.summarize('一。\n二。\n三。\n四。').count(';') == 2  # 只取前3点
assert v1 == v2 and len(v1) == 16            # 确定性、定维
assert llm.calls == 2                          # complete 被调了2次(L30,L34); summarize/embed 不计数
print('✅ MockLLM 就位：确定性补全/摘要/嵌入，可断言、形状贴近真实用法')

## 4 · 立纪律 + 真实适配（无 key 自动回退）

GPU 课用「对拍朴素实现」当裁判；上下文课的裁判是**不变量**：预算器/截断/压缩/检索/缓存在合法输入上做对、满足该满足的约束。

同时，这是本课贯穿始终的 **真实适配骨架**：**有 `ANTHROPIC_API_KEY` 就调真实 Claude（Messages API），没有就自动回退 MockLLM**——所以每本 notebook 都能一路跑到底，绝不阻断。

In [ ]:
import os

def get_llm():
    '''有 key + 装了 anthropic -> 真实 Claude 适配器; 否则 -> MockLLM。
       两者都暴露同一个 .complete(messages) 接口, 上层 scaffold 一行不用改。'''
    if os.environ.get('ANTHROPIC_API_KEY'):
        try:
            import anthropic
            client = anthropic.Anthropic()
            class RealLLM:
                def complete(self, messages):
                    # 真实 Messages API: 把 system 抽出来, 其余作 messages
                    sys_txt = ''.join(m['content'] for m in messages if m['role']=='system')
                    turns = [m for m in messages if m['role'] != 'system']
                    resp = client.messages.create(
                        model='claude-sonnet-4-6', max_tokens=1024,
                        system=sys_txt or None, messages=turns)
                    return ''.join(b.text for b in resp.content if b.type=='text')
            return RealLLM(), 'real(claude-sonnet-4-6)'
        except Exception as e:
            print('真实 API 不可用, 回退 MockLLM:', type(e).__name__)
    return MockLLM(rules=[('天气','今天晴，26°C。')], default='(mock)'), 'mock'

active_llm, kind = get_llm()
print('当前使用的 LLM:', kind)            # 本环境无 key -> mock
ans = active_llm.complete([{'role':'user','content':'今天天气？'}])
print('回答:', ans)
assert kind in ('mock', 'real(claude-sonnet-4-6)')
assert isinstance(ans, str) and len(ans) > 0
print('✅ 适配骨架就位: 有 key 用真实 Claude, 无 key 自动回退 MockLLM —— 绝不阻断')

## 5 · 一个会贯穿全课的小工具：近似 token 计数

本课处处要「数 token」。真实系统用 `messages.count_tokens`（对目标模型精确，且**绝不要用别家 tokenizer**）。本课用一个**确定性近似**：保证可重复、可断言，重点在「计数→预算→裁剪」这条控制流，而非数字精确。

In [ ]:
def count_tokens(text):
    '''确定性近似: 按空白切词 + 长词拆分, 粗估 token 数。
       真实请改用 client.messages.count_tokens(model=..., messages=...)。'''
    if not text:
        return 0
    words = text.split()
    # 经验近似: 平均每个空白分隔词约 1.3 token (长词更多)
    return sum(max(1, (len(w) + 3) // 4) for w in words)

def count_messages(messages):
    '''统计一组消息(role+content)的近似 token, 含每条的角色开销。'''
    total = 0
    for m in messages:
        total += 4  # 每条消息的固定开销(角色标记等)
        total += count_tokens(m['content'] if isinstance(m['content'], str)
                              else json.dumps(m['content'], ensure_ascii=False))
    return total

assert count_tokens('') == 0
assert count_tokens('hello world') >= 2
msgs = [{'role':'system','content':'you are helpful'},
        {'role':'user','content':'what is the capital of france'}]
t = count_messages(msgs)
print('两条消息近似 token:', t)
assert t > 8 and isinstance(t, int)   # 含角色开销, 单调合理
# 单调性: 更长文本 token 不更少
assert count_tokens('a b c d e') >= count_tokens('a b c')
print('✅ 近似计数就位: 可重复、单调、含角色开销 —— 真实换成 count_tokens 即可')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你写的每个 scaffold（预算/截断、compaction、文件记忆、检索、prompt 缓存）都用**不变量 + assert** 验证；逻辑正确则 assert 通过，assert 通过则可把 MockLLM 换成真实 `client.messages.create(model='claude-sonnet-4-6', ...)` 直接迁移（且无 key 自动回退、绝不阻断）。

**接下来五个模块**：01 token 预算与截断 → 02 compaction 与摘要 → 03 文件记忆 → 04 检索入上下文 → 05 prompt 缓存。每一步都建立在「上下文窗口 = 有限预算」这张图上。

下一站：**模块 01 · Token 预算与截断**。